In [126]:
import random_forest_oconnell
import pandas as pd
from sklearn.ensemble import  RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

In [101]:
# CONSTANTS
FEATURE_COLS = [
    "danceability",
    "energy",
    "tempo",
    "valence",
    "acousticness",
    "instrumentalness",
    "liveness",
    "speechiness",
    "loudness",
    "key",
    "mode"
]
REQUIRED_COLS = ["artist", "track"] + FEATURE_COLS
# Create df from big dataset and clean it so cols match and it only has one artist
SONGS = pd.read_csv("data/tracks_features.csv")
SONGS = SONGS.rename(columns={"name": "track"})
SONGS["artist"] = SONGS["artists"].apply(random_forest_oconnell.extract_first_artist)
SONGS = SONGS[REQUIRED_COLS]
SONGS["artist"] = SONGS["artist"].astype(str).str.strip()
SONGS["track"] = SONGS["track"].astype(str).str.strip()

In [10]:
# Loading user data from csv
song_df = pd.read_csv("data/tre_lastfm.csv", sep=",")
song_df.columns = song_df.columns.str.strip().str.lower()
song_df["count"] = 1
song_df = (
    song_df.groupby(["artist", "track"], as_index=False)["count"]
    .sum()
    .sort_values("count", ascending=False)
    .reset_index(drop=True)
)
if len(song_df) > 200:
    song_df = song_df.iloc[:200]    # Trimming df if it's longer than 200 values
features = random_forest_oconnell.obtain_features(song_df)

Searching Spotify for IDs...
Processed 25/200
Processed 50/200
Processed 75/200
Processed 100/200
Processed 125/200
Processed 150/200
Processed 175/200
Processed 200/200
Fetched features for batch 0 (40 songs)
Fetched features for batch 0 (40 songs)
Fetched features for batch 1 (40 songs)
Fetched features for batch 2 (40 songs)
Fetched features for batch 3 (40 songs)


In [63]:
song_df_with_feats = random_forest_oconnell.create_feature_df(song_df, features)

In [43]:
# Saving it in case I need to use it in the future
song_df_with_feats.to_csv("data/tre_lastfm_with_features.csv", index=False)

In [62]:
#Dropping rows with NA values
clean_df = song_df_with_feats.dropna(subset=FEATURE_COLS).copy()

In [45]:
# Creating user profile so we can see what they generally like
user_profile = (
    clean_df[FEATURE_COLS]
    .multiply(clean_df["count"], axis=0)
    .sum() / clean_df["count"].sum()
)

print(user_profile)

danceability          0.610216
energy                0.637947
tempo               123.622726
valence               0.440958
acousticness          0.211092
instrumentalness      0.084349
liveness              0.189028
speechiness           0.066116
loudness             -6.447882
key                   5.241970
mode                  0.563169
dtype: float64


In [103]:
# Now I'm going to remove songs that the user has listened to, so we don't recommend something they've heard already.
heard_songs = clean_df[["artist", "track"]].drop_duplicates().copy()

candidate_df = SONGS.merge(
    heard_songs,
    on=["artist", "track"],
    how="left",
    indicator=True
)
candidate_df = candidate_df[candidate_df["_merge"] == "left_only"].drop(columns=["_merge"])

In [120]:
# Inserting some unheard songs, so the model can better predict.
positive_df = clean_df.copy()
negative_df = candidate_df.sample(n=len(positive_df), random_state=42).copy()
negative_df["count"] = 0

train_df = pd.concat([positive_df, negative_df], ignore_index=True)

X = train_df[FEATURE_COLS]
y = train_df["count"]

In [121]:
# Actually training the model
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)

model.fit(X_train, y_train)

pred = model.predict(X_test)

print("Model evaluation:")
mse = mean_squared_error(y_test, pred)
print("MSE:", mse)

Model evaluation:
MSE: 8.812583499999999


In [122]:
# For reference, here's the model's performance without adding false/noisy values
X = clean_df[FEATURE_COLS]
y = clean_df["count"]
# Actually training the model
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model_with_no_noise = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)

model_with_no_noise.fit(X_train, y_train)

pred = model_with_no_noise.predict(X_test)

print("Model evaluation:")
mse = mean_squared_error(y_test, pred)
print("MSE:", mse)

Model evaluation:
MSE: 28.334917


In [123]:
# Getting recommendations
candidate_X = candidate_df[FEATURE_COLS]
candidate_df["predicted_count"] = model.predict(candidate_X)
recommendations = candidate_df.sort_values(
    "predicted_count",
    ascending=False
)

In [125]:
recommendations.head(20)

,artist,track,danceability,energy,tempo,valence,acousticness,instrumentalness,liveness,speechiness,loudness,key,mode,predicted_count
1071174,Urbs & Cutex,Right Where-!,0.724,0.701,97.407,0.5040,0.00863,0.785000,0.6300,0.0255,-4.894,8,1,13.735
565427,Atlanta Rhythm Section,Who You Gonna Run To,0.691,0.724,109.959,0.6620,0.01110,0.003170,0.6240,0.0261,-4.922,9,1,12.980
310558,Vibronics,Rastaman's Story,0.800,0.762,138.984,0.3920,0.00589,0.883000,0.6080,0.0335,-4.315,10,1,12.705
961431,Soda Stereo,Trátame Suavemente - Remasterizado 2007,0.738,0.789,122.852,0.5390,0.04010,0.064700,0.6320,0.0283,-4.009,11,0,12.425
1001140,Liquid Stranger,Psychonaut,0.695,0.809,120.022,0.0767,0.00705,0.061000,0.6370,0.0526,-6.245,8,1,12.125
738922,Soda Stereo,Trátame Suavemente - Remasterizado 2007,0.738,0.801,122.857,0.5320,0.04380,0.057200,0.5950,0.0284,-3.366,11,0,11.980
668261,Major Lazer,Make It Hot,0.753,0.765,97.000,0.4970,0.00279,0.025000,0.6960,0.0403,-4.648,11,1,11.965
910319,Funky Fool,Honeybadger,0.802,0.738,125.992,0.4460,0.00443,0.773000,0.6370,0.0555,-3.526,8,0,11.950
828279,HALIENE,Dream In Color - Ruben de Ronde Remix,0.687,0.786,128.020,0.4100,0.01000,0.092400,0.6590,0.0839,-4.589,8,0,11.920
630519,Apoptygma Berzerk,Shadow,0.693,0.754,131.988,0.5420,0.00323,0.206000,0.5700,0.0277,-5.104,8,1,11.770
